In [10]:
import torch
import model
import lancedb
import os
from hydra import initialize, initialize_config_module, initialize_config_dir, compose
from omegaconf import OmegaConf
from dotenv import load_dotenv
from datafusion import functions as f
from torch.utils.data import DataLoader, random_split
import torch.nn.functional as F
import numpy as np
from PIL import Image
from torchvision import transforms
import multiprocessing
import trackio

In [11]:
with initialize(version_base=None, config_path="conf"):
    load_dotenv()
    write_url = os.environ["TRACKIO_WRITE_URL"]
    cfg = compose(config_name="config", overrides=[f'trackio.write_url="{write_url}"'])
    trackio.init(name=cfg.trackio.run_name, project=cfg.trackio.project, server_url=cfg.trackio.write_url)

* Run finished. Uploading logs to the remote Trackio server (please wait...)
* Created new run: test-9


In [12]:
vae = model.AutoEncoder(cfg).to(cfg.device)
vae.load_state_dict(torch.load(f'./vae-{cfg.trackio.run_name}.pt', weights_only=True))
vae.eval()

AutoEncoder(
  (conv): Sequential(
    (0): Conv2d(3, 32, kernel_size=(4, 4), stride=(2, 2))
    (1): ReLU()
    (2): Conv2d(32, 64, kernel_size=(4, 4), stride=(2, 2))
    (3): ReLU()
    (4): Conv2d(64, 128, kernel_size=(4, 4), stride=(2, 2))
    (5): ReLU()
    (6): Conv2d(128, 256, kernel_size=(4, 4), stride=(2, 2))
    (7): ReLU()
  )
  (dense): Dense(
    (mu): Linear(in_features=1024, out_features=32, bias=True)
    (log_sigma): Linear(in_features=1024, out_features=32, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=32, out_features=1024, bias=True)
    (1): Fit()
    (2): ConvTranspose2d(1024, 128, kernel_size=(5, 5), stride=(2, 2))
    (3): ReLU()
    (4): ConvTranspose2d(128, 64, kernel_size=(5, 5), stride=(2, 2))
    (5): ReLU()
    (6): ConvTranspose2d(64, 32, kernel_size=(6, 6), stride=(2, 2))
    (7): ReLU()
    (8): ConvTranspose2d(32, 3, kernel_size=(6, 6), stride=(2, 2))
    (9): Sigmoid()
  )
)

In [13]:
rnn = model.RNN(cfg).to(cfg.device)

In [14]:
class FullEpisodicDataset(torch.utils.data.Dataset):
    def __init__(self, cfg):
        super().__init__()
        self.lancedb_uri = cfg.dataset.lancedb_uri
        self.img_size = cfg.dataset.img_size
        self._table = None
        self._len = lancedb.connect(cfg.dataset.lancedb_uri).open_table("episodes").count_rows()

    def _get_table(self):
        if self._table is None:
            self._table = lancedb.connect(self.lancedb_uri).open_table("episodes")
        return self._table

    def __getitem__(self, idx):
        episode = self._get_table().search().where(f'episode_id = {idx}').limit(1).to_arrow()
        frames_np = np.array(episode.column('observations').combine_chunks()[0].as_py(), dtype=np.uint8).reshape(-1, self.img_size, self.img_size, 3)
        actions_np = np.array(episode.column('actions').combine_chunks()[0].as_py(), dtype=np.float32)
        obs = torch.from_numpy(frames_np).clone().permute(0, 3, 1, 2).to(torch.float32) / 255  # (T, 3, H, W)
        acts = torch.from_numpy(actions_np).clone()  # (T, 3)
        return obs, acts

    def __len__(self):
        return self._len

In [15]:
ds = FullEpisodicDataset(cfg)
train_ds, valid_ds = random_split(ds, [0.9, 0.1])

In [16]:
num_workers = min(4, multiprocessing.cpu_count() - 1)
_loader_kwargs = dict(batch_size=1, collate_fn=lambda b: b[0],
                      num_workers=num_workers, persistent_workers=num_workers > 0,
                      prefetch_factor=2 if num_workers > 0 else None,
                      multiprocessing_context='fork' if num_workers > 0 else None)
train_loader = DataLoader(train_ds, **_loader_kwargs)
val_loader = DataLoader(valid_ds, **_loader_kwargs)

In [17]:
with torch.no_grad():
    sample_obs = next(iter(train_loader))[0].to(cfg.device)
    _, _, log_sigma = vae.encode(sample_obs[:10])
    print("mean VAE sigma:", torch.exp(log_sigma).mean().item())
    print("mean VAE sigma per dim:", torch.exp(log_sigma).mean(0))

/Users/matheoledevehat/.local/share/uv/python/cpython-3.11.11-macos-aarch64-none/lib/python3.11/multiprocessing/popen_fork.py:66: RuntimeWarning: lancedb fork support is experimental: the internal async runtime has been reset in the forked child, but a small chance of deadlock remains if other state was mid-operation at fork time. The 'forkserver' or 'spawn' multiprocessing start method is likely a safer alternative.
  self.pid = os.fork()
/Users/matheoledevehat/.local/share/uv/python/cpython-3.11.11-macos-aarch64-none/lib/python3.11/multiprocessing/popen_fork.py:66: RuntimeWarning: lancedb fork support is experimental: the internal async runtime has been reset in the forked child, but a small chance of deadlock remains if other state was mid-operation at fork time. The 'forkserver' or 'spawn' multiprocessing start method is likely a safer alternative.
  self.pid = os.fork()
/Users/matheoledevehat/.local/share/uv/python/cpython-3.11.11-macos-aarch64-none/lib/python3.11/multiprocessing/

mean VAE sigma: 0.8387848138809204
mean VAE sigma per dim: tensor([0.9424, 0.8112, 0.9319, 0.9263, 0.6241, 1.0062, 0.9357, 1.0698, 0.9579,
        0.1544, 0.9314, 0.6115, 0.9873, 0.9480, 0.1858, 0.9396, 0.9514, 0.9936,
        0.9689, 0.9792, 0.9343, 0.9641, 0.9581, 0.5875, 0.7397, 0.4244, 0.8419,
        0.9105, 0.8107, 0.8675, 0.9348, 1.0110], device='mps:0')


Exception ignored in: 

In [18]:
img_size = cfg.dataset.img_size
optimizer = torch.optim.AdamW(rnn.parameters(), cfg.training.rnn.lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, cfg.training.rnn.nb_epochs)

for _ in range(cfg.training.rnn.nb_epochs):
    for obs, actions in train_loader:
        obs = obs.to(cfg.device)      # (T_obs, 3, H, W)
        actions = actions.to(cfg.device)  # (T_act, 3)
        T = min(obs.shape[0], actions.shape[0])

        # Encode all frames with frozen VAE — no gradients needed here
        with torch.no_grad():
            z, _, _ = vae.encode(obs[:T])  # (T, 32)

        # Teacher forcing: feed z[0..T-2] + a[0..T-2], predict distribution over z[1..T-1]
        pi, mu, sigma, _ = rnn(z[:-1], actions[:T-1])       # (T-1, ...) each
        loss = model.MDN.loss(pi, mu, sigma, z[1:].detach())

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        trackio.log({"rnn-training-loss": loss.item()})

    with torch.no_grad():
        for obs, actions in val_loader:
            obs = obs.to(cfg.device)
            actions = actions.to(cfg.device)
            T = min(obs.shape[0], actions.shape[0])
            z, _, _ = vae.encode(obs[:T])
            pi, mu, sigma, _ = rnn(z[:-1], actions[:T-1])
            loss = model.MDN.loss(pi, mu, sigma, z[1:])
            trackio.log({"rnn-val-loss": loss.item()})

    scheduler.step()

KeyboardInterrupt: 

In [ ]:
trackio.finish()

In [ ]:
torch.save(rnn.state_dict(), f'./rnn-{cfg.trackio.run_name}.pt')